# Build an End-to-End System

This puts together the chain of prompts that you saw throughout the course.

## Setup
#### Load the API key and relevant Python libaries.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
import sys

import json
from collections import defaultdict

sys.path.append('../..')
import utils

import panel as pn  # GUI
pn.extension()

load_dotenv(find_dotenv())

client = OpenAI()

def get_completion_from_messages(
    messages,
    model="gpt-4.1-mini",
    temperature=0,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    
    return response.choices[0].message.content

## System of chained prompts for processing the user query

In [2]:
def process_user_message(user_input, all_messages, debug=True):
    delimiter = "```"
    
    # Step 1: Check input to see if it flags the Moderation API or is a prompt injection
    response = client.moderations.create(input=user_input)
    moderation_output = response.results[0]

    if moderation_output.flagged:
        print("Step 1: Input flagged by Moderation API.")
        return "Sorry, we cannot process this request."

    if debug: print("Step 1: Input passed moderation check.")
    
    category_and_product_response = utils.find_category_and_product_only(user_input, utils.get_products_and_category())
    #print(print(category_and_product_response)
    # Step 2: Extract the list of products
    category_and_product_list = utils.read_string_to_list(category_and_product_response)
    #print(category_and_product_list)

    if debug: print("Step 2: Extracted list of products.")

    # Step 3: If products are found, look them up
    product_information = utils.generate_output_string(category_and_product_list)
    if debug: print("Step 3: Looked up product information.")

    # Step 4: Answer the user question
    system_message = f"""
    You are a customer service assistant for a large electronic store. \
    Respond in a friendly and helpful tone, with concise answers. \
    Make sure to ask the user relevant follow-up questions.
    """
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': f"{delimiter}{user_input}{delimiter}"},
        {'role': 'assistant', 'content': f"Relevant product information:\n{product_information}"}
    ]

    final_response = get_completion_from_messages(all_messages + messages)
    if debug:print("Step 4: Generated response to user question.")
    all_messages = all_messages + messages[1:]

    # Step 5: Put the answer through the Moderation API
    response = client.moderations.create(input=final_response)
    moderation_output = response.results[0]

    if moderation_output.flagged:
        if debug: print("Step 5: Response flagged by Moderation API.")
        return "Sorry, we cannot provide this information."

    if debug: print("Step 5: Response passed moderation check.")

    # Step 6: Ask the model if the response answers the initial user query well
    user_message = f"""
    Customer message: {delimiter}{user_input}{delimiter}
    Agent response: {delimiter}{final_response}{delimiter}

    Does the response sufficiently answer the question?
    """
    messages = [
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_message}
    ]
    evaluation_response = get_completion_from_messages(messages)
    if debug: print("Step 6: Model evaluated the response.")

    # Step 7: If yes, use this answer; if not, say that you will connect the user to a human
    if "Y" in evaluation_response:  # Using "in" instead of "==" to be safer for model output variation (e.g., "Y." or "Yes")
        if debug: print("Step 7: Model approved the response.")
        return final_response, all_messages
    else:
        if debug: print("Step 7: Model disapproved the response.")
        neg_str = "I'm unable to provide the information you're looking for. I'll connect you with a human representative for further assistance."
        return neg_str, all_messages

user_input = "tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also what tell me about your tvs"
response,_ = process_user_message(user_input,[])
print(response)

Step 1: Input passed moderation check.
Step 2: Extracted list of products.
Step 3: Looked up product information.
Step 4: Generated response to user question.
Step 5: Response passed moderation check.
Step 6: Model evaluated the response.
Step 7: Model approved the response.
The SmartX ProPhone is a powerful smartphone with a 6.1-inch display, 128GB storage, a 12MP dual camera, and 5G support, priced at $899.99. The FotoSnap DSLR Camera features a 24.2MP sensor, 1080p video, a 3-inch LCD, and interchangeable lenses, priced at $599.99.

For TVs, we have several options:
- CineView 4K TV: 55-inch, 4K resolution, HDR, Smart TV, $599.99
- CineView OLED TV: 55-inch, 4K, HDR, OLED display, $1499.99
- CineView 8K TV: 65-inch, 8K resolution, HDR, Smart TV, $2999.99

We also offer home theater systems and soundbars if you're interested in enhancing your audio experience.

Would you like details on any specific TV model or accessories?


### Chat with the chatbot!
Note that the system message includes detailed instructions about what the OrderBot should do.

In [5]:
panels = []
context = [{'role': 'system', 'content': "You are Service Assistant"}]

inp = pn.widgets.TextInput(placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant", button_type="primary")

conversation = pn.Column(height=300, scroll=True)

def collect_messages(event=None, debug=False):
    global context, panels

    user_input = inp.value_input.strip()
    if debug:
        print(f"User input: {user_input}")

    if not user_input:
        return

  
    inp.value = ""
    inp.value_input = ""

    response, context = process_user_message(user_input, context, debug=debug)


    context.append({'role': 'assistant', 'content': response})

 
    panels.append(
        pn.Row("User:", pn.pane.Markdown(user_input, width=600))
    )
    panels.append(
        pn.Row(
            "Assistant:",
            pn.pane.Markdown(
                response,
                width=600,
                styles={'background': '#F6F6F6', 'padding': '8px', 'border-radius': '6px'}
            )
        )
    )

    conversation.objects = panels


button_conversation.on_click(collect_messages)

dashboard = pn.Column(
    inp,
    button_conversation,
    conversation
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Button(button_type='primary', name='Service Assistant')
    [2] Column(height=300, scroll=True)